In [1]:
%%capture cap
%run "D:\코딩공부\4. 통계\0630\LAB08 EDA 연습문제.ipynb"

In [2]:
### 2. 불러온 내용 확인
print("목표변수 :", target)
print("연속형(종속변수 포함) :", continuous_cols)
print("독립변수(명목형): ", nominal_cols)
print("독립변수(연속형): ", feature_continuous)

display(df.head())              # 품질점검 완료 후 타입변환 데이터프레임
display(df_desc)                   # 기술통계량 데이터프레임
display(df_cat)               # 범주형 데이터 기술통계량 데이터프레임

목표변수 : charges
연속형(종속변수 포함) : ['age', 'bmi', 'children', 'charges']
독립변수(명목형):  ['sex', 'smoker', 'region']
독립변수(연속형):  ['age', 'bmi', 'children']


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.924
1,18,male,33.770,1,no,southeast,1725.552
2,28,male,33.000,3,no,southeast,4449.462
3,33,male,22.705,0,no,northwest,21984.471
4,32,male,28.880,0,no,northwest,3866.855


,count,mean,std,min,25%,50%,75%,max,rel_diff,rdiff_flag,iqr,upper_bound,lower_bound,upper_outliers,upper_outliers_ratio,lower_outliers,lower_outliers_ratio,outliers,outliers_ratio,skew,skew_interpret,kurt,kurt_interpret,log_need
age,1337.000,39.222,14.044,18.000,27.000,39.000,51.000,64.000,0.006,similar,24.000,87.000,-9.000,0,0.000,0,0.000,0,0.000,0.055,symmetric,-1.244,platykurtic,none
bmi,1337.000,30.663,6.100,15.960,26.290,30.400,34.700,53.130,0.009,similar,8.410,47.315,13.675,9,0.007,0,0.000,9,0.007,0.284,symmetric,-0.053,platykurtic,none
children,1337.000,1.096,1.206,0.000,0.000,1.000,2.000,5.000,0.096,similar,2.000,5.000,-3.000,0,0.000,0,0.000,0,0.000,0.937,right tail,0.201,leptokurtic,log1p
charges,1337.000,13279.121,12110.360,1121.874,4746.344,9386.161,16657.717,63770.428,0.415,diff,11911.373,34524.778,-13120.716,139,0.104,0,0.000,139,0.104,1.515,right tail,1.604,leptokurtic,log1p


,Unnamed: 0,sex,smoker,region
0,count,1337,1337,1337
1,unique,2,2,4
2,top,male,no,southeast
3,freq,675,1063,364


In [3]:
# 명목형 라벨링
df1 = my_prep.labeling(df, columns=nominal_cols)
df1.head()

sex (2종): female=0, male=1
smoker (2종): no=0, yes=1
region (4종): northeast=0, northwest=1, southeast=2, southwest=3


,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,3,16884.924
1,18,1,33.770,1,0,2,1725.552
2,28,1,33.000,3,0,2,4449.462
3,33,1,22.705,0,0,1,21984.471
4,32,1,28.880,0,0,1,3866.855


In [4]:
# 더미변수 인코딩
df2 = my_prep.dummies(df1, columns=nominal_cols, drop_first=True)
df2.to_excel("체크포인트A_기준선.xlsx", index=False)
df2.head()

region (4종) -> 3개: region_1, region_2, region_3    (기준: 0 제외)
sex (2종) -> 생략: 이진 변수이므로 원래 컬럼을 유지
smoker (2종) -> 생략: 이진 변수이므로 원래 컬럼을 유지

컬럼 수: 7개 -> 9개


,age,sex,bmi,children,smoker,charges,region_1,region_2,region_3
0,19,0,27.900,0,1,16884.924,0,0,1
1,18,1,33.770,1,0,1725.552,0,1,0
2,28,1,33.000,3,0,4449.462,0,1,0
3,33,1,22.705,0,0,21984.471,1,0,0
4,32,1,28.880,0,0,3866.855,1,0,0


In [4]:
# 로그 변환 대상 컬럼 설정
# 우측꼬리 로그 변환 대상 컬럼
log_cols = df_desc.loc[df_desc['log_need'] == 'log1p'].index.tolist()
log_cols

['children', 'charges']

In [6]:
# 로그 변환 대상 컬럼 설정
# 좌측꼬리 로그 변환 대상 컬럼
reflect_cols = df_desc.loc[df_desc['log_need'] == 'reverse_log1p'].index.tolist()
reflect_cols

[]

In [7]:
# 로그변환
df3 = my_prep.log_transform(df2, log_columns=log_cols,
                            reflect_columns=reflect_cols)
df3.to_excel("체크포인트B_로그변환.xlsx", index=False)
df3.head()

컬럼        꼬리방향      변환식                   역변환식                                  왜도
----------------------------------------------------------------------------------------
children  우측 꼬리     log(1+x)              exp(y)-1                  0.94 -> 0.26
charges   우측 꼬리     log(1+x)              exp(y)-1                 1.51 -> -0.09


,age,sex,bmi,children,smoker,charges,region_1,region_2,region_3
0,19,0,27.900,0.000,1,9.734,0,0,1
1,18,1,33.770,0.693,0,7.454,0,1,0
2,28,1,33.000,1.386,0,8.401,0,1,0
3,33,1,22.705,0.000,0,9.998,1,0,0
4,32,1,28.880,0.000,0,8.260,1,0,0


In [8]:
# 이상치 대체
df4 = my_prep.replace_outlier(df3, columns=feature_continuous)
df4.to_excel("체크포인트C_이상치대체.xlsx", index=False)
df4.head()

이상치 대체 방식: 'bound' (기준: IQR x 1.5)
컬럼                               정상 범위       이상치                   대체값
--------------------------------------------------------------------
age                    -9.00 ~ 87.00    0개(0.0%)      -9.00 또는 87.00
bmi                    13.67 ~ 47.32    9개(0.7%)      13.67 또는 47.32
children                -1.65 ~ 2.75    0개(0.0%)       -1.65 또는 2.75


,age,sex,bmi,children,smoker,charges,region_1,region_2,region_3
0,19,0,27.900,0.000,1,9.734,0,0,1
1,18,1,33.770,0.693,0,7.454,0,1,0
2,28,1,33.000,1.386,0,8.401,0,1,0
3,33,1,22.705,0.000,0,9.998,1,0,0
4,32,1,28.880,0.000,0,8.260,1,0,0


In [9]:
# 다중공선성 제거
df5 = my_prep.reduce_vif(df4, columns=feature_continuous, threshold=10.0)
df5.to_excel("체크포인트D_다중공선성제거.xlsx", index=False)
df5.head()


완료! 남은 변수: ['age', 'bmi', 'children']
최대 VIF = 1.01


,age,bmi,children
0,19,27.900,0.000
1,18,33.770,0.693
2,28,33.000,1.386
3,33,22.705,0.000
4,32,28.880,0.000


In [10]:
# 정규화
df6 = my_prep.scaling(df5, columns=feature_continuous, method='standard')
df6.to_excel("체크포인트E_정규화.xlsx", index=False)
df6.head()

StandardScaler 적용 (3개 컬럼)
컬럼                            변환 전                  변환 후
--------------------------------------------------------
age                18.00 ~ 64.00          -1.51 ~ 1.76
bmi                15.96 ~ 47.32          -2.43 ~ 2.75
children             0.00 ~ 1.79          -1.04 ~ 2.17


,age,bmi,children
0,-1.440,-0.454,-1.043
1,-1.512,0.515,0.199
2,-0.799,0.388,1.441
3,-0.443,-1.312,-1.043
4,-0.514,-0.292,-1.043
